# Voronoi Mosaic Generator

This notebook uses Voronoi tessellation to turn an image into a mosaic of colored cells. It explores random versus edge-aware seeds, stained-glass boundaries, efficient nearest-neighbor lookup, and several ways to assign color to each Voronoi region.

In [ ]:
import sys
import time
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from scipy.spatial import Voronoi, voronoi_plot_2d, KDTree

sys.path.insert(0, '../tools')

try:
    import mosaic_tile_generator as mtg
    print('Imported mosaic_tile_generator from ../tools')
except Exception as exc:
    mtg = None
    print(f'Using inline fallback functions: {exc}')

def make_test_image(size=512):
    x = np.linspace(0, 1, size)
    y = np.linspace(0, 1, size)
    xx, yy = np.meshgrid(x, y)
    base = np.dstack([
        xx,
        yy,
        0.5 + 0.5 * np.sin(2 * np.pi * (xx * 0.75 + yy * 0.5)),
    ])
    image = Image.fromarray((np.clip(base, 0, 1) * 255).astype(np.uint8), mode='RGB')
    draw = ImageDraw.Draw(image, 'RGBA')
    draw.ellipse((60, 60, 220, 220), fill=(255, 120, 80, 180))
    draw.rectangle((280, 80, 460, 220), fill=(60, 180, 255, 170))
    draw.polygon([(120, 400), (256, 260), (390, 430)], fill=(255, 240, 80, 170))
    return image

def random_seeds(count, width, height, rng):
    xs = rng.uniform(0, width - 1, count)
    ys = rng.uniform(0, height - 1, count)
    return np.column_stack([xs, ys])

def edge_weighted_seeds(edge_map, count, rng):
    weights = edge_map.ravel().astype(np.float64)
    weights = weights + weights.mean() * 0.25 + 1e-9
    weights /= weights.sum()
    indices = rng.choice(edge_map.size, size=count, replace=False, p=weights)
    ys, xs = np.unravel_index(indices, edge_map.shape)
    jitter = rng.uniform(-0.45, 0.45, size=(count, 2))
    return np.column_stack([xs, ys]) + jitter

def label_map(shape, seeds, use_kdtree=True):
    h, w = shape[:2]
    yy, xx = np.indices((h, w))
    pts = np.column_stack([xx.ravel(), yy.ravel()])
    if use_kdtree:
        _, idx = KDTree(seeds).query(pts)
    else:
        dist2 = ((pts[:, None, :] - seeds[None, :, :]) ** 2).sum(axis=2)
        idx = dist2.argmin(axis=1)
    return idx.reshape(h, w)

def sample_colors(arr, seeds):
    h, w = arr.shape[:2]
    xs = np.clip(np.round(seeds[:, 0]).astype(int), 0, w - 1)
    ys = np.clip(np.round(seeds[:, 1]).astype(int), 0, h - 1)
    return arr[ys, xs]

def mosaic_from_labels(arr, seeds, labels, mode='sample'):
    colors = sample_colors(arr, seeds).astype(np.float32)
    out = colors[labels].copy()
    if mode == 'average':
        for i in range(len(seeds)):
            mask = labels == i
            if np.any(mask):
                out[mask] = arr[mask].mean(axis=0)
    elif mode == 'dominant':
        for i in range(len(seeds)):
            mask = labels == i
            if np.any(mask):
                vals = arr[mask]
                q = (vals // 32).astype(np.int32)
                keys, counts = np.unique(q, axis=0, return_counts=True)
                out[mask] = np.clip(keys[counts.argmax()] * 32 + 16, 0, 255)
    return out.astype(np.uint8)

def boundary_mask(labels):
    mask = np.zeros_like(labels, dtype=bool)
    mask[:-1, :] |= labels[:-1, :] != labels[1:, :]
    mask[:, :-1] |= labels[:, :-1] != labels[:, 1:]
    return mask

def stained_glass(arr, seeds, labels, line_color=(20, 20, 20)):
    out = mosaic_from_labels(arr, seeds, labels, mode='average').copy()
    mask = boundary_mask(labels)
    out[mask] = np.array(line_color, dtype=np.uint8)
    return out

## Creating Test Images

A colorful synthetic image makes it easy to see how Voronoi cells simplify shape and color. Smooth gradients show how cell size controls detail, while bold geometric overlays make boundaries and sampling strategies more obvious.

In [ ]:
test_image = make_test_image(512)
test_array = np.asarray(test_image)

plt.figure(figsize=(6, 6))
plt.imshow(test_array)
plt.title('Colorful synthetic test image')
plt.axis('off')
plt.show()

## Random Voronoi Tessellation - explain Voronoi diagrams

A Voronoi diagram partitions the plane into regions around seed points so that every location belongs to its nearest seed. In mosaic terms, each cell becomes a tile whose footprint is defined by proximity rather than a fixed square grid.

In [ ]:
rng = np.random.default_rng(12)
seeds_50 = random_seeds(50, test_array.shape[1], test_array.shape[0], rng)
vor = Voronoi(seeds_50)

fig = voronoi_plot_2d(vor, show_vertices=False, line_colors='black', line_width=1.2, line_alpha=0.8, point_size=20)
plt.gca().set_xlim(0, test_array.shape[1])
plt.gca().set_ylim(test_array.shape[0], 0)
plt.gca().set_aspect('equal')
plt.title('Voronoi tessellation with 50 random seeds')
plt.show()

## Voronoi Mosaic - sampling image at seed points

The simplest mosaic renderer colors each cell using the source image color at the seed itself. That makes the region flat-colored but still image-aware, which is often enough to create a convincing faceted look.

In [ ]:
labels_50 = label_map(test_array.shape, seeds_50, use_kdtree=True)
mosaic_50 = mosaic_from_labels(test_array, seeds_50, labels_50, mode='sample')

plt.figure(figsize=(6, 6))
plt.imshow(mosaic_50)
plt.title('Voronoi mosaic with 50 sampled-color seeds')
plt.axis('off')
plt.show()

## Effect of Seed Count - more seeds = more detail

Seed count controls the abstraction level. Few seeds create a very blocky, posterized mosaic, while many seeds preserve more of the original image structure and local color transitions.

In [ ]:
counts = [10, 25, 50, 100, 200]
rng = np.random.default_rng(4)
results = []
for count in counts:
    seeds = random_seeds(count, test_array.shape[1], test_array.shape[0], rng)
    labels = label_map(test_array.shape, seeds, use_kdtree=True)
    results.append((count, mosaic_from_labels(test_array, seeds, labels, mode='sample')))

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, (count, mosaic) in zip(axes, results):
    ax.imshow(mosaic)
    ax.set_title(f'{count} seeds')
    ax.axis('off')
plt.tight_layout()
plt.show()

## Cell Boundaries as Stained Glass

Explicit borders turn the mosaic into a stained-glass style rendering. Dark separator lines emphasize geometry and help individual cells read as separate pieces rather than simple flat fills.

In [ ]:
seeds_glass = random_seeds(80, test_array.shape[1], test_array.shape[0], np.random.default_rng(33))
labels_glass = label_map(test_array.shape, seeds_glass, use_kdtree=True)
glass = stained_glass(test_array, seeds_glass, labels_glass)

plt.figure(figsize=(6, 6))
plt.imshow(glass)
plt.title('Stained-glass Voronoi mosaic')
plt.axis('off')
plt.show()

## Edge-Weighted Seed Distribution

Uniformly random seeds do not know where the image has important structure. By concentrating more seeds near high gradients, we preserve detail along edges and shape boundaries while spending fewer cells on smooth, low-information areas.

In [ ]:
gray = test_array.mean(axis=2).astype(np.float32)
gy, gx = np.gradient(gray)
edge_mag = np.hypot(gx, gy)

rng = np.random.default_rng(8)
random_seed_set = random_seeds(120, test_array.shape[1], test_array.shape[0], rng)
weighted_seed_set = edge_weighted_seeds(edge_mag, 120, rng)

random_labels = label_map(test_array.shape, random_seed_set, use_kdtree=True)
weighted_labels = label_map(test_array.shape, weighted_seed_set, use_kdtree=True)
random_mosaic = mosaic_from_labels(test_array, random_seed_set, random_labels, mode='sample')
weighted_mosaic = mosaic_from_labels(test_array, weighted_seed_set, weighted_labels, mode='sample')

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
axes[0, 0].imshow(edge_mag, cmap='magma')
axes[0, 0].scatter(random_seed_set[:, 0], random_seed_set[:, 1], s=8, c='white')
axes[0, 0].set_title('Random seeds over gradient magnitude')
axes[0, 0].axis('off')
axes[0, 1].imshow(edge_mag, cmap='magma')
axes[0, 1].scatter(weighted_seed_set[:, 0], weighted_seed_set[:, 1], s=8, c='white')
axes[0, 1].set_title('Edge-weighted seeds over gradient magnitude')
axes[0, 1].axis('off')
axes[1, 0].imshow(random_mosaic)
axes[1, 0].set_title('Random-seed mosaic')
axes[1, 0].axis('off')
axes[1, 1].imshow(weighted_mosaic)
axes[1, 1].set_title('Edge-weighted mosaic')
axes[1, 1].axis('off')
plt.tight_layout()
plt.show()

## KD-Tree Efficient Computation

The expensive part of a Voronoi mosaic is repeated nearest-seed lookup. A KD-tree accelerates those queries dramatically compared with a brute-force distance check, especially as seed count and image resolution increase.

In [ ]:
sample = test_array[::2, ::2]
seeds_perf = random_seeds(150, sample.shape[1], sample.shape[0], np.random.default_rng(21))

t0 = time.perf_counter()
labels_brute = label_map(sample.shape, seeds_perf, use_kdtree=False)
brute_time = time.perf_counter() - t0

t1 = time.perf_counter()
labels_kd = label_map(sample.shape, seeds_perf, use_kdtree=True)
kd_time = time.perf_counter() - t1

print(f'Brute-force time: {brute_time:.3f} s')
print(f'KD-tree time: {kd_time:.3f} s')
print('Same label map:', np.array_equal(labels_brute, labels_kd))

plt.figure(figsize=(6, 4))
plt.bar(['Brute force', 'KD-tree'], [brute_time, kd_time], color=['#c44e52', '#4c72b0'])
plt.ylabel('Seconds')
plt.title('Nearest-neighbor timing comparison')
plt.show()

## Color Variations

Once the cell geometry is fixed, color assignment changes the visual character. Sampling the seed gives a crisp faceted look, averaging gives smoother tonal massing, and dominant-color assignment pushes the result toward a posterized stained-glass palette.

In [ ]:
seeds_color = random_seeds(60, test_array.shape[1], test_array.shape[0], np.random.default_rng(55))
labels_color = label_map(test_array.shape, seeds_color, use_kdtree=True)

sampled = mosaic_from_labels(test_array, seeds_color, labels_color, mode='sample')
averaged = mosaic_from_labels(test_array, seeds_color, labels_color, mode='average')
dominant = mosaic_from_labels(test_array, seeds_color, labels_color, mode='dominant')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, title, image in zip(axes, ['Sampled seed color', 'Average cell color', 'Dominant cell color'], [sampled, averaged, dominant]):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()